# 00. Start here: how to think about image processing in Python

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner  
**Goal:** Build the mental model used in every later notebook.

## What you will learn

- What an image is inside Python
- Why NumPy is central to image processing
- The difference between **displaying**, **processing**, **segmenting**, and **measuring**
- How to inspect an image before choosing an algorithm
- The course workflow: **question → load → inspect → preprocess → segment → validate → measure → save**

> **Main rule:** Do not begin by asking “Which function should I run?” Begin by asking “What problem am I trying to solve?”


## 1. The simplest possible mental model

A grayscale image is a rectangular table of numbers. Each number is a pixel intensity.

```text
           columns →
         0    1    2
row 0   10   40   90
row 1   15   70  120
row 2   20   80  200
```

Python usually stores that table as a **NumPy array**.

A color or multichannel image usually adds another axis:

```text
(rows, columns, channels)
```

This means image processing is not mysterious: most operations are ways of **selecting, transforming, comparing, or measuring array values**.


## 2. Four words that beginners should keep separate

### Display
Make the image visible to a human.  
Example: `plt.imshow(image)`

### Preprocess
Modify the image to reduce an unwanted effect or prepare for a later step.  
Examples: smoothing, background correction, contrast transformation.

### Segment
Decide which pixels or regions belong to the structure of interest.  
Examples: thresholding, watershed.

### Measure
Calculate properties **after** the structure has been identified.  
Examples: area, mean intensity, centroid.

A very common mistake is to improve the display and then assume the data itself has improved scientifically. These are different tasks.


## 3. The workflow used throughout this course

```text
Question
   ↓
Load
   ↓
Inspect
   ↓
Preprocess only if needed
   ↓
Segment
   ↓
Validate
   ↓
Measure
   ↓
Save results + parameters
```

### Why validation is in the middle

If the segmentation is wrong, the measurements can be mathematically precise and still scientifically wrong.

So the workflow is **not**:

```text
image → algorithm → spreadsheet
```

It is:

```text
image → inspect → choose → test → validate → quantify
```


## 4. How comments are written in this course

Comments focus on decisions:

- **Why** are we using this function?
- **When** is it appropriate?
- Which **parameter** changes its behavior?
- What should we **inspect afterward**?

We do not comment trivial syntax just to make the notebook look longer. The goal is readable, professional scientific code.


In [ ]:
# sys gives access to information about the Python interpreter.
import sys

# NumPy stores images as arrays and performs numerical operations.
import numpy as np

# Matplotlib displays images and plots.
import matplotlib
import matplotlib.pyplot as plt

# scikit-image provides image-processing algorithms and teaching images.
import skimage as ski

# pandas will later organize measurement results into tables.
import pandas as pd

# imageio will later read and write common image files.
import imageio

# Why print versions?
# If analysis is repeated months later, software versions help explain differences
# and are part of a reproducible computational workflow.
print("Python       :", sys.version.split()[0])
print("NumPy        :", np.__version__)
print("Matplotlib   :", matplotlib.__version__)
print("scikit-image :", ski.__version__)
print("pandas       :", pd.__version__)
print("imageio      :", imageio.__version__)


## 5. Load your first image

For the first lessons we use `skimage.data`.

### Why use built-in images?

- everyone gets the same input,
- no file-path problems,
- no external image licensing is required for the lesson,
- we can focus on the processing concept.

Later you will use the same ideas with your own images.


In [ ]:
# ski.data.camera() returns a 2-D grayscale teaching image as a NumPy array.
image = ski.data.camera()

# Always inspect basic properties before processing.
print("Python type     :", type(image))
print("shape           :", image.shape)
print("number of axes  :", image.ndim)
print("dtype           :", image.dtype)
print("minimum         :", image.min())
print("maximum         :", image.max())
print("mean intensity  :", image.mean())

# cmap="gray" tells Matplotlib to display a 2-D intensity image in grayscale.
# This changes the display style, not the stored image values.
plt.figure(figsize=(6, 5))
plt.imshow(image, cmap="gray")
plt.title("First image")
plt.axis("off")
plt.tight_layout()
plt.show()


## 6. How to read the inspection results

Suppose you see:

```text
shape: (512, 512)
dtype: uint8
range: 0 to 255
```

Interpretation:

- `512 × 512` → 512 rows and 512 columns.
- `uint8` → unsigned 8-bit integer storage.
- `0–255` → a common range for an 8-bit image.

### Why this matters

The same numerical threshold does not mean the same thing for every dtype and range.  
A value of `100` is reasonable inside a `uint8` image, but not inside a float image normalized to `0–1`.


## 7. A small function you will reuse: `describe_image`

Instead of rewriting the same inspection lines, we can package them into a function.

### When should you create a function?

Create one when a group of steps:
- has a clear single purpose,
- is repeated,
- benefits from consistent behavior.


In [ ]:
def describe_image(image, name="image"):
    """Print basic image properties before processing.

    Use this function at the beginning of an analysis or whenever a new
    intermediate array has a dtype/range that you do not understand.
    """
    print(f"{name}")
    print("  shape :", image.shape)
    print("  ndim  :", image.ndim)
    print("  dtype :", image.dtype)
    print("  min   :", image.min())
    print("  max   :", image.max())
    print("  mean  :", float(image.mean()))


# Test the helper.
describe_image(image, "camera")


## 8. Function map: where should I look first?

| Question | Tool | Why |
|---|---|---|
| What shape/range/type is this image? | NumPy array attributes/statistics | Understand the data first |
| How do I display it? | `matplotlib.pyplot` | Visualization |
| How do I change contrast? | `ski.exposure` | Intensity transformations |
| How do I smooth or find edges? | `ski.filters` | Filtering |
| How do I clean a binary mask? | `ski.morphology` | Shape/size operations |
| How do I label or measure objects? | `ski.measure` | Object-level analysis |
| How do I separate regions? | `ski.segmentation` | Segmentation algorithms |

You do not need to memorize the whole scikit-image API. Learn the **category of problem** first.


## 9. Recommended project organization

```text
project/
├── data/
│   ├── raw/          # copies of untouched inputs
│   └── processed/    # optional derived images
├── notebooks/        # exploration and teaching
├── outputs/
│   ├── figures/
│   └── tables/
├── requirements.txt
└── README.md
```

### Raw-data rule

Do not overwrite the only copy of experimental data while learning or testing code.


## Common beginner mistakes

1. Running a filter before understanding the image dtype/range.
2. Measuring before checking whether segmentation is correct.
3. Changing five parameters at the same time.
4. Confusing a nicer display with better quantitative data.
5. Copying code without understanding what problem the function solves.

## Practice

Replace:

```python
image = ski.data.camera()
```

with:

```python
image = ski.data.coins()
```

Then answer:

1. What is its shape?
2. What is its dtype?
3. What is its range?
4. Is it grayscale or multichannel?
5. What would you inspect before choosing a threshold?

## Takeaway

**An image is an array. A good analysis is a sequence of justified decisions made on that array.**
